In [9]:
import sys
from pathlib import Path

# path_to_your_local_folder = Path("/Users/marco/Work-MBP/gtfs_railways")
path_to_your_local_folder = Path("C:/Users/KIIT/Documents/UAntwerp/railways_resilience")
sys.path.append(str(path_to_your_local_folder))

from utils.imports import *

In [10]:
# Belgium
from config import PATH_TO_SQLITE
attributes_be = load_gtfs(str(PATH_TO_SQLITE))
L_graph_be = load_graph(DATA_DIR / "pkl/belgium_routesCleaned.pkl")

# Netherlands
attributes_nl = load_gtfs(str(DATA_DIR / "sqlite/NL.sqlite"))
L_graph_nl=load_graph(DATA_DIR / "pkl/nl_merged.pkl")

In [11]:
plot_graph(L_graph_be, back_map="OSM")

In [12]:
import pandas as pd
# results_csv = DATA_DIR / "results/removal_runs_targeted_node_BE/targeted_removal_seed42_nodes558.csv"
results_csv = DATA_DIR / "results/removal_runs_targeted_node_NL/targeted_removal_seed42_nodes390.csv"

df = pd.read_csv(results_csv)
df.head()

,step,removed_node,normalized_efficiency,percent_remaining,removal_time_seconds
0,0,NaN,1.000000,100.000000,0.0000
1,1,52.0,0.852941,99.743590,1100.3415
2,2,9.0,0.762657,99.487179,855.2641
3,3,3.0,0.700199,99.230769,807.8185
4,4,90.0,0.517423,98.974359,883.4747


In [20]:
removal_order = df["removed_node"].iloc[:-1].tolist()

In [21]:
# plot_graph(L_graph_be, back_map="OSM")
plot_graph(L_graph_nl, back_map="OSM")

In [22]:
# ======================================================
# Cell 2 — Selenium driver (reuse for all exports)
# ======================================================
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service)

print("✅ Selenium Chrome driver started.")


✅ Selenium Chrome driver started.


In [23]:
# ======================================================
# Cell 3 — Patch: export with tiles by waiting + using driver
# ======================================================
import time
from pathlib import Path

# frames_dir = Path("frames_bokeh_osm_stride5")
frames_dir = Path("frames_bokeh_osm_stride5_nl")
frames_dir.mkdir(exist_ok=True)

def save_graph_frame_osm(G, step, export_wait=1.5):
    export_stub = str(frames_dir / f"frame_{step:06d}")
    # Build the plot object, then export with webdriver (small wait helps tiles load)
    p = plot_graph_return_figure(G, back_map="OSM")  # we'll define this next cell
    time.sleep(export_wait)
    export_png(p, filename=export_stub + ".png", webdriver=driver)

print("✅ Frame saver stub ready (needs plot_graph_return_figure defined next).")


✅ Frame saver stub ready (needs plot_graph_return_figure defined next).


In [24]:
# ======================================================
# Cell 4 — Copy of plot_graph that RETURNS the figure
# ======================================================
from bokeh.plotting import figure
from bokeh.models import HoverTool, Circle, MultiLine, LinearColorMapper, ColorBar, WheelZoomTool
from bokeh.tile_providers import get_provider, Vendors
from bokeh.plotting import from_networkx
from pyproj import Transformer
import networkx as nx

def plot_graph_return_figure(G, space="L", back_map=False, color_by="", edge_color_by=""):
    p = figure(height=600, width=950, toolbar_location='below',
               tools="pan, wheel_zoom, box_zoom, reset, save")

    # Build node position dict
    pos_dict = {}
    transformer = Transformer.from_crs("epsg:4326", "epsg:3857", always_xy=True)
    for i, d in G.nodes(data=True):
        if back_map == "OSM":
            x2, y2 = transformer.transform(float(d["lon"]), float(d["lat"]))
        else:
            x2, y2 = float(d["lon"]), float(d["lat"])
        pos_dict[i] = (x2, y2)

    graph = from_networkx(G, layout_function=pos_dict)

    # Hover tools
    node_hover_tool = HoverTool(tooltips=[("index", "@index"), ("name", "@name")], renderers=[graph.node_renderer])
    edge_tooltips = [("duration_avg", "@duration_avg")] if space == "L" else [("avg_wait", "@avg_wait")]
    hover_edges = HoverTool(tooltips=edge_tooltips, renderers=[graph.edge_renderer], line_policy="interp")
    p.add_tools(node_hover_tool, hover_edges)

    # Node coloring
    if color_by and all(color_by in d for _, d in G.nodes(data=True)):
        vals = nx.get_node_attributes(G, color_by).values()
        mapper = LinearColorMapper(palette="RdYlGn11", low=min(vals), high=max(vals))
        graph.node_renderer.glyph = Circle(size=7, fill_color={'field': color_by, 'transform': mapper})
    else:
        graph.node_renderer.glyph = Circle(size=7)

    # Edge coloring
    if edge_color_by and all(edge_color_by in d for _, _, d in G.edges(data=True)):
        edge_vals = [d[edge_color_by] for _, _, d in G.edges(data=True)]
        mapper = LinearColorMapper(palette="RdYlGn11", low=min(edge_vals), high=max(edge_vals))
        graph.edge_renderer.glyph = MultiLine(line_width=4, line_alpha=0.5,
                                             line_color={'field': edge_color_by, 'transform': mapper})
        color_bar = ColorBar(color_mapper=mapper, label_standoff=12, border_line_color=None, location=(0, 0))
        p.add_layout(color_bar, "right")
    else:
        graph.edge_renderer.glyph = MultiLine(line_width=4, line_alpha=0.5)

    graph.node_renderer.selection_glyph = Circle(fill_color='blue')
    graph.node_renderer.hover_glyph = Circle(fill_color='red')

    p.toolbar.active_scroll = p.select_one(WheelZoomTool)

    if space == "P":
        graph.edge_renderer.selection_glyph = MultiLine(line_color='black', line_width=5)
        graph.edge_renderer.hover_glyph = MultiLine(line_color='black', line_width=10)
    else:
        graph.edge_renderer.selection_glyph = MultiLine(line_color='blue', line_width=5)
        graph.edge_renderer.hover_glyph = MultiLine(line_color='red', line_width=5)

    p.renderers.append(graph)

    if back_map == "OSM":
        p.add_tile(get_provider(Vendors.CARTODBPOSITRON))

    return p

print("✅ plot_graph_return_figure() defined.")


✅ plot_graph_return_figure() defined.


In [25]:
# ======================================================
# Cell 5 — Frame saver that exports OSM tiles
# ======================================================
import time
from bokeh.io import export_png

def save_graph_frame_osm(G, step, export_wait=1.5):
    export_stub = str(frames_dir / f"frame_{step:06d}")
    p = plot_graph_return_figure(G, space="L", back_map="OSM")
    time.sleep(export_wait)  # tiles load
    export_png(p, filename=export_stub + ".png", webdriver=driver)

print("✅ OSM frame saver ready.")


✅ OSM frame saver ready.


In [26]:
# ======================================================
# Cell 6 — Run removals, save frames with OSM basemap
# ======================================================
# G = L_graph_be.copy()
G = L_graph_nl.copy()

def disconnect_node_edges_only(G, node):
    if node not in G:
        return 0

    # explicit in/out removal (handles directed + multigraph)
    if G.is_multigraph():
        edges_to_remove = []
        if G.is_directed():
            edges_to_remove += list(G.out_edges(node, keys=True))
            edges_to_remove += list(G.in_edges(node, keys=True))
        else:
            edges_to_remove += list(G.edges(node, keys=True))
    else:
        edges_to_remove = []
        if G.is_directed():
            edges_to_remove += list(G.out_edges(node))
            edges_to_remove += list(G.in_edges(node))
        else:
            edges_to_remove += list(G.edges(node))

    G.remove_edges_from(edges_to_remove)
    return len(edges_to_remove)

stride = 5
saved = 0

for step, node in enumerate(removal_order, start=1):
    disconnect_node_edges_only(G, node)

    if step % stride == 0:
        save_graph_frame_osm(G, step=step, export_wait=1.5)
        saved += 1

    if step % 20 == 0:
        print(f"Progress {step}/{len(removal_order)} | saved frames: {saved} | edges remaining: {G.number_of_edges()}")

print("✅ Done exporting frames with OSM basemap.")


Progress 20/390 | saved frames: 4 | edges remaining: 704


Progress 40/390 | saved frames: 8 | edges remaining: 588


Progress 60/390 | saved frames: 12 | edges remaining: 488


Progress 80/390 | saved frames: 16 | edges remaining: 410


Progress 100/390 | saved frames: 20 | edges remaining: 338


...

Progress 320/390 | saved frames: 64 | edges remaining: 4


Progress 340/390 | saved frames: 68 | edges remaining: 2


Progress 360/390 | saved frames: 72 | edges remaining: 2


Progress 380/390 | saved frames: 76 | edges remaining: 0


✅ Done exporting frames with OSM basemap.


In [27]:
# ======================================================
# Cell 7 — Stitch into GIF
# ======================================================
import imageio.v2 as imageio
from pathlib import Path

# gif_path = Path(f"belgium_disconnect_osm_stride{stride}.gif")
gif_path = Path(f"netherlands_disconnect_osm_stride{stride}.gif")
frame_files = sorted(frames_dir.glob("frame_*.png"))

with imageio.get_writer(gif_path, mode="I", duration=0.15) as writer:
    for fp in frame_files:
        writer.append_data(imageio.imread(fp))

print("✅ GIF created:", gif_path.resolve())


✅ GIF created: C:\Users\KIIT\Documents\UAntwerp\railways_resilience\notebooks\Marco\netherlands_disconnect_osm_stride5.gif


In [81]:
# ======================================================
# Cell 8 — Cleanup
# ======================================================
driver.quit()
print("✅ Closed Selenium driver.")


✅ Closed Selenium driver.
